# Carga de datos de Twitch en InfluxDB


## 1. Instalar librerías


In [ ]:
# Instala las librerías necesarias para leer los CSV y conectar con InfluxDB.
%pip install influxdb-client pandas python-dotenv


## 2. Configuración


In [ ]:
# Importa las librerías que se usan en el notebook.
import csv
import os
import re

import pandas as pd
from dotenv import load_dotenv
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

# Carga las credenciales desde el .env de PIA.
load_dotenv("../../PIA/.env")

# InfluxDB corre en Docker y publica el puerto 8086 en el host.
URL = os.getenv("INFLUXDB_URL", "http://localhost:8086")
TOKEN = os.getenv("DOCKER_INFLUXDB_INIT_ADMIN_TOKEN")
ORG = os.getenv("DOCKER_INFLUXDB_INIT_ORG")
BUCKET = "TwitchData"

# Muestra la configuración sin enseñar el token.
print("URL:", URL)
print("Organización:", ORG)
print("Bucket:", BUCKET)


## 3. Conexión y bucket


In [ ]:
# Crea la conexión con InfluxDB.
client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

# Comprueba que InfluxDB responde.
print("Conexión:", client.ping())

# Crea el bucket TwitchData solo si no existe.
buckets_api = client.buckets_api()
if buckets_api.find_bucket_by_name(BUCKET) is None:
    buckets_api.create_bucket(bucket_name=BUCKET, org=ORG)
    print(f"Bucket '{BUCKET}' creado.")
else:
    print(f"Bucket '{BUCKET}' ya existe.")


## 4. Datos mensuales


In [ ]:
# Lee el CSV mensual. El separador es ';' y los valores llevan espacios.
df = pd.read_csv(
    "twitch_games_monthly_historical.csv",
    sep=";",
    skipinitialspace=True,
)
df.columns = df.columns.str.strip()
df["Game"] = df["Game"].str.strip()

# Convierte la fecha del mes a UTC.
df["DateTime"] = pd.to_datetime(df["DateTime"], utc=True)

# Crea un punto por cada juego y mes.
puntos = [
    Point("twitch_games_monthly")
    .tag("game", fila.Game)
    .time(fila.DateTime, WritePrecision.S)
    .field("average_viewers", int(fila.AverageViewers))
    .field("peak_viewers", int(fila.PeakViewers))
    .field("hours_watched", int(fila.HoursWatched))
    .field("hours_streamed", int(fila.HoursStreamed))
    .field("average_channels", int(fila.AverageChannels))
    .field("peak_channels", int(fila.PeakChannels))
    for fila in df.itertuples(index=False)
]

# Escribe los puntos en InfluxDB.
write_api.write(bucket=BUCKET, org=ORG, record=puntos)
print(f"Mensual: {len(puntos)} puntos.")


## 5. Emisiones de streamers


In [ ]:
# Lee el CSV de streamers. GamesPlayed puede contener ';' dentro del campo.
with open("twitch_streamer_broadcasts_combined.csv", encoding="utf-8-sig", newline="") as archivo:
    lector = csv.reader(archivo, delimiter=";")
    cabecera = [columna.strip() for columna in next(lector)]
    filas = []
    for fila in lector:
        if not fila:
            continue
        fila = [valor.strip() for valor in fila]
        filas.append(fila[:10] + [";".join(fila[10:]).strip('"')])

# Convierte las filas en una tabla.
df = pd.DataFrame(filas, columns=cabecera[:10] + ["GamesPlayed"])

# Convierte las fechas de inicio y fin a UTC.
df["StreamStart"] = pd.to_datetime(df["StreamStart"], utc=True)
df["StreamEnd"] = pd.to_datetime(df["StreamEnd"], utc=True)

# Convierte "6 hours, 30 mins" a minutos.
def duracion_minutos(texto):
    horas = re.search(r"(\d+)\s*hours?", texto)
    minutos = re.search(r"(\d+)\s*mins?", texto)
    return (int(horas.group(1)) if horas else 0) * 60 + (int(minutos.group(1)) if minutos else 0)

df["duration_minutes"] = df["Duration"].apply(duracion_minutos)

# Cuenta cuántos juegos se jugaron en cada emisión.
df["games_played_count"] = df["GamesPlayed"].apply(
    lambda texto: len([juego for juego in texto.split(";") if juego.strip()])
)

# Crea un punto por cada emisión.
puntos = [
    Point("twitch_streamer_broadcasts")
    .tag("streamer", fila.Streamer)
    .time(fila.StreamStart, WritePrecision.S)
    .field("watch_time_hours", int(fila.WatchTimeHours))
    .field("average_viewers", int(fila.AvgViewers))
    .field("peak_viewers", int(fila.PeakViewers))
    .field("followers_gained", int(fila.FollowersGained))
    .field("followers_per_hour", int(fila.FollowersPerHour))
    .field("duration_minutes", int(fila.duration_minutes))
    .field("games_played", fila.GamesPlayed)
    .field("games_played_count", int(fila.games_played_count))
    for fila in df.itertuples(index=False)
]

# Escribe los puntos en InfluxDB.
write_api.write(bucket=BUCKET, org=ORG, record=puntos)
print(f"Emisiones: {len(puntos)} puntos.")


## 6. Juegos por streamer


In [ ]:
# Crea un punto por cada juego de cada emisión.
puntos_juegos = []
for fila in df.itertuples(index=False):
    # Separa los juegos que se jugaron en la emisión.
    juegos = [juego.strip() for juego in fila.GamesPlayed.split(";") if juego.strip()]
    if not juegos:
        continue
    # Reparte las horas vistas a partes iguales entre los juegos.
    horas = int(fila.WatchTimeHours)
    parte = horas / len(juegos)
    for juego in juegos:
        puntos_juegos.append(
            Point("twitch_streamer_games")
            .tag("streamer", fila.Streamer)
            .tag("game", juego)
            .time(fila.StreamStart, WritePrecision.S)
            .field("watch_time_hours_share", float(parte))
            .field("watch_time_hours", horas)
            .field("broadcasts", 1)
            .field("average_viewers", int(fila.AvgViewers))
            .field("peak_viewers", int(fila.PeakViewers))
            .field("followers_gained", int(fila.FollowersGained))
            .field("duration_minutes", int(fila.duration_minutes))
        )

# Escribe los puntos en InfluxDB.
write_api.write(bucket=BUCKET, org=ORG, record=puntos_juegos)
print(f"Streamers por juego: {len(puntos_juegos)} puntos.")


## 7. Top 365 días y cierre


In [ ]:
# Lee el CSV del top 365d.
df = pd.read_csv(
    "twitch_top_games_365d.csv",
    sep=";",
    skipinitialspace=True,
)
df.columns = df.columns.str.strip()
df["Game"] = df["Game"].str.strip()

# El CSV no trae fecha; se usa la fecha de la descarga.
snapshot = pd.Timestamp("2026-10-21", tz="UTC")

# Crea un punto por cada juego del top.
puntos = [
    Point("twitch_top_games_365d")
    .tag("game", fila.Game)
    .tag("snapshot_date", "2026-10-21")
    .time(snapshot, WritePrecision.S)
    .field("rank", int(fila.Rank))
    .field("watch_time_hours", int(fila.WatchTimeHours))
    .field("stream_time_hours", int(fila.StreamTimeHours))
    .field("peak_viewers", int(fila.PeakViewers))
    .field("peak_channels", int(fila.PeakChannels))
    .field("streamers", int(fila.Streamers))
    .field("average_viewers", int(fila.AverageViewers))
    .field("average_channels", int(fila.AverageChannels))
    .field("average_viewer_ratio", float(fila.AverageViewerRatio))
    for fila in df.itertuples(index=False)
]

# Escribe los puntos en InfluxDB.
write_api.write(bucket=BUCKET, org=ORG, record=puntos)
print(f"Top 365d: {len(puntos)} puntos.")

# Cierra la conexión. No borra el bucket ni los datos.
client.close()
print(f"Datos cargados en el bucket {BUCKET}.")
